# Register Tiling GEMM - Nsight Systems & Nsight Compute Profiling

**使用前请确保：** Runtime → Change runtime type → GPU (T4)

本 notebook 会：
1. 编译并运行 register tiling matmul kernel
2. 使用 **Nsight Systems (`nsys`)** 进行系统级时间线分析
3. 使用 **Nsight Compute (`ncu`)** 进行 kernel 级深度分析

## nsys vs ncu 的区别

| | Nsight Systems (`nsys`) | Nsight Compute (`ncu`) |
|---|---|---|
| **层级** | 系统级 / 应用级 | Kernel 级 |
| **看什么** | 整体时间线：CPU/GPU 交互、kernel launch 开销、内存拷贝、API 调用、空闲间隙 | 单个 kernel 内部：occupancy、bank conflict、warp stall、计算/访存吞吐 |
| **用途** | 找宏观瓶颈（"时间花在哪了"） | 找微观瓶颈（"kernel 为什么慢"） |
| **开销** | 轻量，适合全程 profile | 重量，逐 kernel 深入分析 |
| **典型流程** | **第一步** — 先用 nsys 定位瓶颈 kernel / 阶段 | **第二步** — 再用 ncu 深挖那个 kernel |

In [1]:
# 检查 GPU 型号和 CUDA 版本
!nvidia-smi
!nvcc --version

Sun Apr  5 00:56:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
%%writefile matmul_reg_tiling.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 8
#define TM 8
#define TN 8

__global__ void MatMulRegTilingKernel(const float* __restrict__ A,
                                       const float* __restrict__ B,
                                       float* __restrict__ C,
                                       int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;

    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    __shared__ float As[BM][BK];
    __shared__ float Bs[BK][BN];

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;

    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;

    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    for (int k = 0; k < K; k += BK) {
        for (int i = 0; i < BM; i += strideA) {
            As[loadARow + i][loadACol] = aPtr[(loadARow + i) * K + k + loadACol];
        }
        for (int i = 0; i < BK; i += strideB) {
            Bs[loadBRow + i][loadBCol] = bPtr[(k + loadBRow + i) * N + loadBCol];
        }

        __syncthreads();

        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm) {
                regA[tm] = As[ty * TM + tm][bk];
            }
            for (int tn = 0; tn < TN; ++tn) {
                regB[tn] = Bs[bk][tx * TN + tn];
            }
            for (int tm = 0; tm < TM; ++tm) {
                for (int tn = 0; tn < TN; ++tn) {
                    regC[tm][tn] += regA[tm] * regB[tn];
                }
            }
        }

        __syncthreads();
    }

    for (int tm = 0; tm < TM; ++tm) {
        for (int tn = 0; tn < TN; ++tn) {
            int globalRow = cRow + ty * TM + tm;
            int globalCol = cCol + tx * TN + tn;
            C[globalRow * N + globalCol] = regC[tm][tn];
        }
    }
}

void MatMulCPU(const float* A, const float* B, float* C, int M, int N, int K)
{
    for (int i = 0; i < M; ++i)
        for (int j = 0; j < N; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < K; ++k)
                sum += A[i * K + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

int main()
{
    // Increased matrix size to 2048
    int M = 2048, N = 2048, K = 2048;

    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    float* A     = (float*)malloc(bytesA);
    float* B     = (float*)malloc(bytesB);
    float* C_gpu = (float*)malloc(bytesC);
    float* C_cpu = (float*)malloc(bytesC);

    for (int i = 0; i < M * K; ++i) A[i] = (float)(i % 10) * 0.1f;
    for (int i = 0; i < K * N; ++i) B[i] = (float)(i % 7)  * 0.1f;

    // GPU
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytesA);
    cudaMalloc(&d_B, bytesB);
    cudaMalloc(&d_C, bytesC);
    cudaMemcpy(d_A, A, bytesA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytesB, cudaMemcpyHostToDevice);

    dim3 dimBlock(BN / TN, BM / TM);
    dim3 dimGrid(N / BN, M / BM);

    // Warmup
    MatMulRegTilingKernel<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaDeviceSynchronize();

    // Timed run
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);
    MatMulRegTilingKernel<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float kernel_ms = 0;
    cudaEventElapsedTime(&kernel_ms, start, stop);

    cudaMemcpy(C_gpu, d_C, bytesC, cudaMemcpyDeviceToHost);

    // CPU validation (Skipped or limited for 2048 to save time, here we keep it but it might be slow)
    // MatMulCPU(A, B, C_cpu, M, N, K);

    double gflops = 2.0 * M * N * K / (kernel_ms * 1e-3) / 1e9;

    printf("=== Register Tiling GEMM (Matrix 2048) ===\n");
    printf("Matrix: %dx%dx%d\n", M, N, K);
    printf("GPU kernel: %.3f ms  (%.1f GFLOPS)\n", kernel_ms, gflops);

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(A); free(B); free(C_gpu); free(C_cpu);
    return 0;
}

Overwriting matmul_reg_tiling.cu


In [27]:
!nvcc -O3 -lineinfo -arch=sm_75 matmul_reg_tiling.cu -o matmul_reg_tiling
print("编译成功!")

编译成功!


In [7]:
# 先正常运行，确认正确性
!./matmul_reg_tiling

=== Register Tiling GEMM ===
Matrix: 1024x1024x1024
Block tile: 128x128, Thread tile: 8x8, K tile: 8
GPU kernel: 1.259 ms  (1705.1 GFLOPS)
Max error: 1.525879e-05
PASSED


---
## Part 1: Nsight Systems (`nsys`) — 系统级时间线分析

`nsys` 用于全局视角，回答：
- 时间主要花在 kernel 执行、内存拷贝、还是 CPU 端？
- kernel launch 有没有间隙/气泡？
- H2D / D2H 传输占多大比例？

In [8]:
# 挂载 Google Drive 并加载 nsys
from google.colab import drive
drive.mount('/content/drive')
!chmod -R +x /content/drive/MyDrive/cuda/nsys_tool

NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} --version

Mounted at /content/drive
NVIDIA Nsight Systems version 2024.5.1.113-245134619542v0


In [9]:
# 1. nsys 基础 profile — 查看 GPU 活动时间线摘要
#    显示: CUDA API 调用时间、kernel 执行时间、内存拷贝时间
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} profile \
     --stats=true \
     --force-overwrite=true \
     -o nsys_matmul_reg_tiling \
     ./matmul_reg_tiling

=== Register Tiling GEMM ===
Matrix: 1024x1024x1024
Block tile: 128x128, Thread tile: 8x8, K tile: 8
GPU kernel: 1.461 ms  (1469.6 GFLOPS)
Max error: 1.525879e-05
PASSED
Generating '/tmp/nsys-report-ca17.qdstrm'
[1/8] [========================100%] nsys_matmul_reg_tiling.nsys-rep
[2/8] [========================100%] nsys_matmul_reg_tiling.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_matmul_reg_tiling.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  ----------------------
     95.5    3,971,746,630         48  82,744,721.5  100,155,657.5     1,775  363,093,664  57,115,185.5  poll                  
      4.4      181,863,714        529     343,787.7       15,700.0     1,007   17,534,724 

In [17]:
# 2. 单独查看 kernel 执行统计
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} stats --force-export=true --report cuda_gpu_kern_sum nsys_matmul_reg_tiling.nsys-rep

Generating SQLite file nsys_matmul_reg_tiling.sqlite from nsys_matmul_reg_tiling.nsys-rep
Processing [nsys_matmul_reg_tiling.sqlite] with [/content/drive/MyDrive/cuda/nsys_tool/host-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                                     Name                                    
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  ---------------------------------------------------------------------------
    100.0        2,890,053          2  1,445,026.5  1,445,026.5  1,443,698  1,446,355      1,878.8  MatMulRegTilingKernel(const float *, const float *, float *, int, int, int)



In [18]:
# 3. 内存拷贝统计 — H2D/D2H 传输耗时和带宽
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} stats --force-export=true --report cuda_gpu_mem_size_sum nsys_matmul_reg_tiling.nsys-rep

Generating SQLite file nsys_matmul_reg_tiling.sqlite from nsys_matmul_reg_tiling.nsys-rep
Processing [nsys_matmul_reg_tiling.sqlite] with [/content/drive/MyDrive/cuda/nsys_tool/host-linux-x64/reports/cuda_gpu_mem_size_sum.py]... 

 ** CUDA GPU MemOps Summary (by Size) (cuda_gpu_mem_size_sum):

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
      8.389      2     4.194     4.194     4.194     4.194        0.000  [CUDA memcpy Host-to-Device]
      4.194      1     4.194     4.194     4.194     4.194        0.000  [CUDA memcpy Device-to-Host]



In [19]:
# 4. CUDA API 调用统计 — 查看 cudaMalloc/cudaMemcpy/cudaLaunchKernel 等耗时
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} stats --force-export=true --report cuda_api_sum nsys_matmul_reg_tiling.nsys-rep

Generating SQLite file nsys_matmul_reg_tiling.sqlite from nsys_matmul_reg_tiling.nsys-rep
Processing [nsys_matmul_reg_tiling.sqlite] with [/content/drive/MyDrive/cuda/nsys_tool/host-linux-x64/reports/cuda_api_sum.py]... 

 ** CUDA API Summary (cuda_api_sum):

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)    Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  ---------  -----------  -------------  ----------------------
     94.7      188,938,408          3  62,979,469.3     88,190.0     70,626  188,779,592  108,946,102.4  cudaMalloc            
      2.5        4,976,991          3   1,658,997.0  1,045,482.0    967,487    2,964,022    1,130,857.4  cudaMemcpy            
      0.8        1,635,844          2     817,922.0    817,922.0     15,467    1,620,377    1,134,842.7  cudaLaunchKernel      
      0.7        1,446,383          1   1,446,383.0  1,446,383.0  1,446,383    1,446,383            

In [20]:
# 5. 导出 .nsys-rep 报告（可下载到本地用 Nsight Systems GUI 打开查看时间线）
print("nsys 报告已保存为 nsys_matmul_reg_tiling.nsys-rep")
print("下载后用 Nsight Systems GUI 打开，可查看完整 CPU/GPU 时间线视图")

from google.colab import files
files.download('nsys_matmul_reg_tiling.nsys-rep')

nsys 报告已保存为 nsys_matmul_reg_tiling.nsys-rep
下载后用 Nsight Systems GUI 打开，可查看完整 CPU/GPU 时间线视图


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### nsys 输出解读

`nsys` 的统计报告会显示几张表：

| 报告表 | 看什么 | 关注点 |
|--------|--------|--------|
| **CUDA Kernel Summary** | 每个 kernel 的执行次数、平均/最大耗时 | 哪个 kernel 最耗时？warmup 和正式运行差别大吗？ |
| **CUDA Memory Operation** | H2D/D2H 拷贝的大小和带宽 | 传输占总时间的比例。如果占比大，考虑 overlap 或 pinned memory |
| **CUDA API Summary** | cudaMalloc/cudaMemcpy/cudaLaunchKernel 等 CPU 端耗时 | cudaMalloc 首次调用很慢（CUDA context 初始化），这是正常的 |

如果下载 `.nsys-rep` 到本地用 GUI 打开，可以看到完整的时间线：
- **横轴**是时间，**纵轴**分 CPU 行和 GPU 行
- 可以直观看到 kernel 和 memcpy 是否有重叠、有没有 GPU 空闲间隙

---
## Part 2: Nsight Compute (`ncu`) — Kernel 级深度分析

`ncu` 深入单个 kernel 内部，回答：
- 计算管线利用率如何？是 compute bound 还是 memory bound？
- occupancy 是多少？受什么限制（寄存器/shared memory）？
- 有没有 shared memory bank conflict？
- warp 停顿的主要原因是什么？

In [22]:
!which ncu && ncu --version

/usr/local/cuda/bin/ncu
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


In [24]:
!ncu --set full \
     --kernel-name MatMulRegTilingKernel \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling

==PROF== Connected to process 13031 (/content/matmul_reg_tiling)
==PROF== Profiling "MatMulRegTilingKernel": 0%....50%....100% - 31 passes
=== Register Tiling GEMM ===
Matrix: 1024x1024x1024
Block tile: 128x128, Thread tile: 8x8, K tile: 8
GPU kernel: 6885.080 ms  (0.3 GFLOPS)
Max error: 1.525879e-05
PASSED
==PROF== Disconnected from process 13031
[13031] matmul_reg_tiling@127.0.0.1
  MatMulRegTilingKernel(const float *, const float *, float *, int, int, int) (8, 8, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.98
    Elapsed Cycles                cycle    1,276,175
    Memory Throughput                 %        37.16
    DRAM Throughput                   %         3.38
    Dura

In [28]:
!ncu --set full \
     --kernel-name MatMulRegTilingKernel \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling

==PROF== Connected to process 14602 (/content/matmul_reg_tiling)
==PROF== Profiling "MatMulRegTilingKernel": 0%....50%....100% - 31 passes
=== Register Tiling GEMM (Matrix 2048) ===
Matrix: 2048x2048x2048
GPU kernel: 8140.409 ms  (2.1 GFLOPS)
==PROF== Disconnected from process 14602
[14602] matmul_reg_tiling@127.0.0.1
  MatMulRegTilingKernel(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    8,713,304
    Memory Throughput                 %        43.04
    DRAM Throughput                   %         4.61
    Duration                         ms        14.89
    L1/TEX Cache Th

In [29]:
!ncu --section MemoryWorkloadAnalysis \
     --section MemoryWorkloadAnalysis_Chart \
     --section MemoryWorkloadAnalysis_Tables \
     --kernel-name MatMulRegTilingKernel \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling

==PROF== Connected to process 14909 (/content/matmul_reg_tiling)
==PROF== Profiling "MatMulRegTilingKernel": 0%....50%....100% - 27 passes
=== Register Tiling GEMM (Matrix 2048) ===
Matrix: 2048x2048x2048
GPU kernel: 5879.587 ms  (2.9 GFLOPS)
==PROF== Disconnected from process 14909
[14909] matmul_reg_tiling@127.0.0.1
  MatMulRegTilingKernel(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Memory Workload Analysis
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    Memory Throughput     Gbyte/s        14.70
    Mem Busy                    %        43.10
    Max Bandwidth               %        28.94
    L1/TEX Hit Rate             %        20.14
    L2 Hit Rate                 %        81.15
    Mem Pipes Busy              %        28.94
    ----------------- ----------- ------------

    Section: Memory Workload A

In [30]:
!ncu --section Occupancy \
     --kernel-name MatMulRegTilingKernel \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling

==PROF== Connected to process 15001 (/content/matmul_reg_tiling)
==PROF== Profiling "MatMulRegTilingKernel": 0%....50%....100% - 1 pass
=== Register Tiling GEMM (Matrix 2048) ===
Matrix: 2048x2048x2048
GPU kernel: 809.936 ms  (21.2 GFLOPS)
==PROF== Disconnected from process 15001
[15001] matmul_reg_tiling@127.0.0.1
  MatMulRegTilingKernel(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           16
    Block Limit Registers                 block            2
    Block Limit Shared Mem                block            4
    Block Limit Warps                     block            4
    Theoretical Active Warps per SM        warp           16
    Theoretical Occupancy      

In [35]:
# 5. Compute Workload 分析 — 查看计算管线利用率
!ncu --section ComputeWorkloadAnalysis \
     --kernel-name MatMulRegTilingKernel \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling

==PROF== Connected to process 16127 (/content/matmul_reg_tiling)
==PROF== Profiling "MatMulRegTilingKernel": 0%....50%....100% - 7 passes
=== Register Tiling GEMM (Matrix 2048) ===
Matrix: 2048x2048x2048
GPU kernel: 2227.995 ms  (7.7 GFLOPS)
==PROF== Disconnected from process 16127
[16127] matmul_reg_tiling@127.0.0.1
  MatMulRegTilingKernel(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Compute Workload Analysis
    -------------------- ----------- ------------
    Metric Name          Metric Unit Metric Value
    -------------------- ----------- ------------
    Executed Ipc Active   inst/cycle         1.25
    Executed Ipc Elapsed  inst/cycle         1.13
    Issue Slots Busy               %        31.37
    Issued Ipc Active     inst/cycle         1.25
    SM Busy                        %        45.18
    -------------------- ----------- ------------

    INF   FMA is the highest-utilized pipeline (45

In [34]:
!ncu --section SchedulerStats \
     --kernel-name MatMulRegTilingKernel \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling

==PROF== Connected to process 15331 (/content/matmul_reg_tiling)
==PROF== Profiling "MatMulRegTilingKernel": 0%....50%....100% - 1 pass
=== Register Tiling GEMM (Matrix 2048) ===
Matrix: 2048x2048x2048
GPU kernel: 930.076 ms  (18.5 GFLOPS)
==PROF== Disconnected from process 15331
[15331] matmul_reg_tiling@127.0.0.1
  MatMulRegTilingKernel(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Scheduler Statistics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    One or More Eligible                   %        31.30
    Issued Warp Per Scheduler                        0.31
    No Eligible                            %        68.70
    Active Warps Per Scheduler          warp         3.83
    Eligible Warps Per Scheduler        warp         0.42
    ---------------------------- -----------

In [36]:
# 7. 导出 .ncu-rep 报告文件（可下载到本地用 Nsight Compute GUI 打开）
!ncu --set full \
     --kernel-name MatMulRegTilingKernel \
     --launch-skip 1 --launch-count 1 \
     -o matmul_reg_tiling_profile \
     ./matmul_reg_tiling

print("\n报告已保存为 matmul_reg_tiling_profile.ncu-rep")
print("可下载后用 Nsight Compute GUI 打开查看完整可视化分析")

==PROF== Connected to process 18173 (/content/matmul_reg_tiling)
==PROF== Profiling "MatMulRegTilingKernel": 0%....50%....100% - 31 passes
=== Register Tiling GEMM (Matrix 2048) ===
Matrix: 2048x2048x2048
GPU kernel: 8304.687 ms  (2.1 GFLOPS)
==PROF== Disconnected from process 18173
==PROF== Report: /content/matmul_reg_tiling_profile.ncu-rep

报告已保存为 matmul_reg_tiling_profile.ncu-rep
可下载后用 Nsight Compute GUI 打开查看完整可视化分析


In [37]:
# 下载 .ncu-rep 文件到本地
from google.colab import files
files.download('matmul_reg_tiling_profile.ncu-rep')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 关键指标解读指南

运行完上面的 profiling 后，重点关注以下指标：

| 指标 | 含义 | 期望值 |
|------|------|--------|
| **Compute (SM) Throughput** | SM 计算利用率 | 越高越好，>70% 为佳 |
| **Memory Throughput** | 显存带宽利用率 | 计算密集型 kernel 此值不宜过高 |
| **Achieved Occupancy** | 实际 warp 占用率 | 本 kernel 因寄存器多，可能 <50%，但不一定是瓶颈 |
| **Registers Per Thread** | 每线程寄存器数 | 预期 ~80+，过多会限制 occupancy |
| **Shared Memory Per Block** | 每 block shared memory | 预期 ~5KB (As 4KB + Bs 4KB) |
| **L1/L2 Hit Rate** | 缓存命中率 | shared memory tiling 应使 L1 压力较低 |
| **Shared Bank Conflicts** | bank conflict 次数 | As 列优先访问可能产生 conflict |
| **Warp Stall Reasons** | warp 停顿原因 | 关注 stall_mio_throttle (shared mem) |

### Register Tiling 的特点
- **低 occupancy 但高性能**：每线程用 80+ 寄存器，occupancy 会较低，
  但寄存器 tiling 大幅减少 shared memory 访问，整体吞吐反而更高
- **计算密集型**：应该看到 compute throughput >> memory throughput
- **可能的优化方向**：如果看到 shared memory bank conflict 较多，
  可以考虑对 As 做 padding（`As[BM][BK+1]`）来消除